# Digital Twin UI — Full Pipeline Notebook

**Purpose:** One-stop notebook for building a surrogate model from simulation results.

## What this notebook does

1. **Discover** completed simulation results (`.xplt` files) in the `runs/` directory
2. **Extract** per-facet contact pressure data using `xplt-parser`
3. **Build** a combined training dataset (CSV format)
4. **Export** a reference facets file (geometry for CSAR computation)
5. **Train** a neural network surrogate model using `surrogate-lab`
6. **Evaluate** model performance and save artifacts
7. **Deploy** artifacts so the LibreChat API can use them immediately

## Prerequisites

- At least one completed simulation in `runs/` (each containing `results.xplt`)
- The matching `.feb` file for each simulation (in `base_configuration/` or same run dir)
- Environment set up via `bash scripts/setup-common-env.sh` (local) or Docker (auto)

## After training

The surrogate model will be available in LibreChat via:
- `compute_csar_vs_depth()` — CSAR vs insertion depth per Z band
- `evaluate_contact_pressure()` — contact pressure at given depths
- `predict_vtp_contact_pressure()` — annotate VTP files with predictions

---
## Cell 0: Environment Check

In [1]:
import sys, os
from pathlib import Path

# ── Resolve paths ────────────────────────────────────────────────────────────
# This notebook works both:
#   • Locally (from repo root): REPO_ROOT = ./
#   • In Docker Jupyter:        REPO_ROOT = /workspace  (bind-mounted)

# When running in Docker, the CWD is /workspace/notebooks.
# Notebooks dir → parent = /workspace  (repo root equivalent)
NOTEBOOK_DIR = Path(os.getcwd())
if NOTEBOOK_DIR.name == 'notebooks':
    REPO_ROOT = NOTEBOOK_DIR.parent
else:
    REPO_ROOT = NOTEBOOK_DIR  # already at root

XPLT_PARSER_DIR = REPO_ROOT / 'xplt-parser'
SURROGATE_LAB_DIR = REPO_ROOT / 'surrogate-lab'

# Add to path so imports work even without pip install
for p in [str(XPLT_PARSER_DIR), str(SURROGATE_LAB_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'Repository root: {REPO_ROOT}')
print(f'xplt-parser:     {XPLT_PARSER_DIR} (exists={XPLT_PARSER_DIR.exists()})')
print(f'surrogate-lab:   {SURROGATE_LAB_DIR} (exists={SURROGATE_LAB_DIR.exists()})')
print()

# Verify imports
try:
    import xplt_core
    print('✓ xplt_core importable')
except ImportError as e:
    print(f'✗ xplt_core import failed: {e}')

try:
    from src.utils.config import load_config
    print('✓ surrogate-lab importable')
except ImportError as e:
    print(f'✗ surrogate-lab import failed: {e}')

try:
    import torch
    print(f'✓ PyTorch {torch.__version__} (device: {"CUDA" if torch.cuda.is_available() else "CPU"})')
except ImportError:
    print('✗ PyTorch not installed')

try:
    import mlflow
    print(f'✓ MLflow {mlflow.__version__}')
except ImportError:
    print('✗ MLflow not installed')

Repository root: /workspace
xplt-parser:     /workspace/xplt-parser (exists=True)
surrogate-lab:   /workspace/surrogate-lab (exists=True)

✓ xplt_core importable
✓ surrogate-lab importable
✓ PyTorch 2.11.0+cu130 (device: CPU)
✓ MLflow 3.10.1


---
## Cell 1: Configuration

**Edit this cell** to match your setup before running the notebook.

In [2]:
import os
from pathlib import Path

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  TOGGLE — flip this boolean to switch data source
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
USE_CUSTOM_PATH = True   # True  → use CUSTOM_SIMULATION_PATH below
                          # False → use the default runs/ directory

# ── Custom path (used when USE_CUSTOM_PATH = True) ────────────────────────────
# Point this to any folder that directly contains .feb and .xplt files,
# OR a folder whose sub-directories each contain .feb/.xplt files.
CUSTOM_SIMULATION_PATH = Path(os.getenv("MANUAL_SIMS_PATH", "/workspace/manual_sims"))

# ── Default path (used when USE_CUSTOM_PATH = False) ─────────────────────────
# Simulation outputs produced by LibreChat / the API (batch sub-directories).
DEFAULT_SIMULATION_PATH = REPO_ROOT / "runs"

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SIMULATION_SOURCES = (
    [CUSTOM_SIMULATION_PATH] if USE_CUSTOM_PATH else [DEFAULT_SIMULATION_PATH]
)

# ── xplt / feb file naming ────────────────────────────────────────────────────
XPLT_FILENAME = None   # None = auto-detect any *.xplt in the folder
FEB_FILENAME  = None   # None = auto-detect any *.feb  in the folder

# Contact surface name substring (None = auto-detect primary surface).
CONTACT_SURFACE_NAME = None

# ── Output / MLflow ───────────────────────────────────────────────────────────
SURROGATE_DATA_DIR = Path(os.getenv("SURROGATE_DATA_PATH",
                                     str(REPO_ROOT / "data" / "surrogate")))
TRAINING_DIR  = SURROGATE_DATA_DIR / "training"
MODELS_DIR    = SURROGATE_DATA_DIR / "models" / "latest"
RESULTS_DIR   = SURROGATE_DATA_DIR / "results"

for d in [TRAINING_DIR, MODELS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def _detect_mlflow_uri(local_fallback: str) -> str:
    """Return a working MLflow tracking URI.

    Priority:
      1. MLFLOW_TRACKING_URI env var (explicit override)
      2. http://mlflow:5000        (Docker-internal hostname)
      3. http://localhost:5000     (MLflow exposed on the host)
      4. local filesystem path     (no server running at all)
    """
    env = os.getenv("MLFLOW_TRACKING_URI")
    if env:
        return env
    import socket
    for host in ("mlflow", "localhost"):
        try:
            socket.create_connection((host, 5000), timeout=1).close()
            return f"http://{host}:5000"
        except OSError:
            pass
    return local_fallback

MLFLOW_URI = _detect_mlflow_uri(str(REPO_ROOT / "data" / "mlruns"))
MLFLOW_EXPERIMENT = "surrogate-lab"
RUN_NAME          = None   # Optional: give this training run a descriptive name

COMBINED_CSV         = TRAINING_DIR / "combined.csv"
REFERENCE_FACETS_CSV = TRAINING_DIR / "reference_facets.csv"

# ── Summary ───────────────────────────────────────────────────────────────────
mode = "CUSTOM PATH" if USE_CUSTOM_PATH else "DEFAULT (runs/)"
print(f"Mode: {mode}")
print(f"  SIMULATION_SOURCES:")
for src in SIMULATION_SOURCES:
    status = "EXISTS" if Path(src).exists() else "NOT FOUND"
    print(f"    [{status}]  {src}")
print(f"  SURROGATE_DATA_DIR:   {SURROGATE_DATA_DIR}")
print(f"  TRAINING_DIR:         {TRAINING_DIR}")
print(f"  MODELS_DIR:           {MODELS_DIR}")
print(f"  MLFLOW_URI:           {MLFLOW_URI}")


Mode: CUSTOM PATH
  SIMULATION_SOURCES:
    [EXISTS]  /workspace/manual_sims
  SURROGATE_DATA_DIR:   /workspace/data/surrogate
  TRAINING_DIR:         /workspace/data/surrogate/training
  MODELS_DIR:           /workspace/data/surrogate/models/latest
  MLFLOW_URI:           http://mlflow:5000


---
## Cell 2: Discover Simulation Results

Scans all paths listed in `SIMULATION_SOURCES`.  Each path is checked in two modes:

* **Direct mode** – the path itself contains `.xplt` / `.feb` files (single run).
* **Batch mode** – the path contains sub-directories, each of which is a run.

You can mix both modes in the same list.


In [3]:
import json
from pathlib import Path

def _resolve_single_run(run_dir: Path, xplt_name: str,
                         feb_name=None) -> dict | None:
    """Try to build a case dict from a folder that directly holds the files."""
    # Support explicit xplt filename OR any *.xplt in the folder
    xplt_candidates = (
        list(run_dir.glob("*.xplt")) if xplt_name is None
        else ([run_dir / xplt_name] if (run_dir / xplt_name).exists()
              else list(run_dir.glob("*.xplt")))
    )
    if not xplt_candidates:
        return None
    xplt_path = xplt_candidates[0]

    # Resolve .feb
    feb_path = None
    if feb_name is not None:
        if (run_dir / feb_name).exists():
            feb_path = run_dir / feb_name
    else:
        run_febs = list(run_dir.glob("*.feb"))
        if run_febs:
            feb_path = run_febs[0]

    # Read metadata if available
    meta = {}
    meta_file = run_dir / "metadata.json"
    if meta_file.exists():
        try:
            meta = json.loads(meta_file.read_text())
        except Exception:
            pass

    return {
        "run_id":       run_dir.name,
        "run_dir":      run_dir,
        "feb_path":     feb_path,
        "xplt_path":    xplt_path,
        "xplt_size_MB": xplt_path.stat().st_size / 1e6,
        "metadata":     meta,
        "source":       str(run_dir.parent),
    }


def find_simulation_cases(sources, xplt_name: str = "input.xplt",
                           feb_name=None):
    """
    Discover simulation cases from a list of source paths.

    Each source path is inspected in two ways:
      1. Direct mode  — source itself contains .xplt files (single run).
      2. Batch mode   — source contains sub-folders that each hold .xplt files.

    Returns a deduplicated list of case dicts, sorted by run_id.
    """
    seen_xplts = set()
    cases = []

    for src in sources:
        src = Path(src)
        if not src.exists():
            print(f"  WARNING: source path does not exist — skipping: {src}")
            continue

        # --- Direct mode: src itself has .xplt files --------------------------
        direct = _resolve_single_run(src, xplt_name, feb_name)
        if direct and direct["xplt_path"] not in seen_xplts:
            seen_xplts.add(direct["xplt_path"])
            cases.append(direct)
            continue          # no need to also scan sub-dirs

        # --- Batch mode: scan sub-directories ---------------------------------
        sub_dirs = sorted(p for p in src.iterdir() if p.is_dir())
        for run_dir in sub_dirs:
            c = _resolve_single_run(run_dir, xplt_name, feb_name)
            if c and c["xplt_path"] not in seen_xplts:
                seen_xplts.add(c["xplt_path"])
                cases.append(c)

    return cases


cases = find_simulation_cases(SIMULATION_SOURCES, XPLT_FILENAME, FEB_FILENAME)

print(f"Found {len(cases)} completed simulation(s):\n")
for c in cases:
    feb_info = c["feb_path"].name if c["feb_path"] else "NOT FOUND"
    print(f"  {c['run_id']:40s}  {c['xplt_size_MB']:6.1f} MB  .feb={feb_info}")
    print(f"  {'':40s}  source: {c['source']}")

if len(cases) == 0:
    print("No completed simulations found.")
    print("Check that SIMULATION_SOURCES points to folders containing .xplt files.")
elif any(c["feb_path"] is None for c in cases):
    n_missing = sum(1 for c in cases if c["feb_path"] is None)
    print(f"\nWARNING: {n_missing} run(s) have no matching .feb file.")
    print("Insertion depth cannot be computed without a .feb file.")
    print("Place a .feb file alongside the .xplt, or set FEB_FILENAME above.")

Found 1 completed simulation(s):

  manual_sims                               2265.5 MB  .feb=sample.feb
                                            source: /workspace


---
## Cell 3: Extract Surrogate Data from xplt Files

This cell uses `xplt_core.SimulationCase` to parse each `.xplt` file and
extract per-facet contact pressure data in a format suitable for training.

In [4]:
import xplt_core
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# Select cases with .feb files (required for insertion depth computation)
valid_cases = [c for c in cases if c['feb_path'] is not None]
print(f'Processing {len(valid_cases)} case(s) with .feb files ...')
print()

all_surrogate_dfs = []
all_facet_dfs = []
loaded_cases = []

for c in valid_cases:
    run_id  = c['run_id']
    try:
        print(f'Loading {run_id} ({c["xplt_size_MB"]:.1f} MB) ...', end=' ')
        sc = xplt_core.SimulationCase(
            feb_path=c['feb_path'],
            xplt_path=c['xplt_path'],
            label=run_id,
            contact_surface_name=CONTACT_SURFACE_NAME,
        )
        print(f'OK  ({sc.n_facets} facets, {sc.n_timesteps} timesteps, '
              f'depth: {sc.insertion_depths.max():.1f} mm)')

        # Extract surrogate training data (tidy long format)
        df_surr = sc.df_surrogate()
        df_surr['run_id'] = run_id  # tag source run
        all_surrogate_dfs.append(df_surr)

        # Extract facet geometry summary
        df_fac = sc.df_facets.copy()
        df_fac['run_id'] = run_id
        all_facet_dfs.append(df_fac)

        loaded_cases.append(sc)

    except Exception as e:
        print(f'FAILED: {e}')

print()
print(f'Successfully loaded {len(loaded_cases)} / {len(valid_cases)} case(s).')

if not all_surrogate_dfs:
    raise RuntimeError(
        'No cases loaded successfully. Check that .xplt and .feb files are '
        'valid and accessible. See error messages above.'
    )

# Quick stats
total_rows = sum(len(df) for df in all_surrogate_dfs)
print(f'Total training samples: {total_rows:,}')

Processing 1 case(s) with .feb files ...

Loading manual_sims (2265.5 MB) ... [manual_sims] Parsing sample.feb …
[manual_sims] Reading sample.xplt …
[manual_sims] Parsing 180 timesteps …
[manual_sims] Ready  |  23734 facets  |  180 timesteps  |  t = [0.000 … 30.350] s  |  max cp = 0.0169 MPa  |  insertion depth = [0.00 … 273.31] mm
OK  (23734 facets, 180 timesteps, depth: 273.3 mm)

Successfully loaded 1 / 1 case(s).
Total training samples: 4,272,120


---
## Cell 4: Build and Save Training Dataset

In [5]:
# Combine all cases into one DataFrame
combined_df = pd.concat(all_surrogate_dfs, ignore_index=True)

print('Combined dataset shape:', combined_df.shape)
print()
print(combined_df.describe())
print()

# Save combined training CSV
combined_df.to_csv(COMBINED_CSV, index=False)
print(f'✓ Training dataset saved: {COMBINED_CSV}')
print(f'  {len(combined_df):,} rows × {len(combined_df.columns)} columns')
print(f'  Size: {COMBINED_CSV.stat().st_size / 1e6:.1f} MB')
print()

# Save per-case CSVs as well (for reference)
for i, (df_s, c) in enumerate(zip(all_surrogate_dfs, valid_cases)):
    per_case_path = TRAINING_DIR / f"{c['run_id']}_surrogate.csv"
    df_s.to_csv(per_case_path, index=False)
    print(f'  ✓ {per_case_path.name}')

print()
print('Preview of training data:')
combined_df.head(5)

Combined dataset shape: (4272120, 7)

         centroid_x    centroid_y    centroid_z    facet_area  \
count  4.272120e+06  4.272120e+06  4.272120e+06  4.272120e+06   
mean   4.420410e-01 -2.865641e-01  1.930731e+02  2.423832e-01   
std    1.592155e+00  1.604374e+00  1.195211e+02  5.387735e-02   
min   -1.892603e+00 -2.607577e+00 -2.846780e-01  2.019382e-02   
25%   -1.094266e+00 -1.864852e+00  8.604368e+01  2.311632e-01   
50%    4.239056e-01 -2.185817e-01  1.911540e+02  2.530316e-01   
75%    2.017020e+00  1.281502e+00  2.975525e+02  2.700840e-01   
max    2.697459e+00  1.989099e+00  4.024081e+02  4.646965e-01   

       insertion_depth  contact_pressure  
count     4.272120e+06      4.272120e+06  
mean      1.494696e+02      4.833137e-04  
std       7.742135e+01      1.049584e-03  
min       0.000000e+00      0.000000e+00  
25%       8.656911e+01      0.000000e+00  
50%       1.547122e+02      0.000000e+00  
75%       2.135372e+02      3.061882e-04  
max       2.733059e+02      1.69

,centroid_x,centroid_y,centroid_z,facet_area,insertion_depth,contact_pressure,run_id
0,-0.365146,-2.227906,2.832700,0.187719,0.0,0.0,manual_sims
1,1.075712,-2.374332,2.554793,0.177791,0.0,0.0,manual_sims
2,-0.841743,0.670698,0.307375,0.318930,0.0,0.0,manual_sims
3,-1.161272,0.368586,11.753905,0.156565,0.0,0.0,manual_sims
4,-1.239121,-0.130011,11.437157,0.177645,0.0,0.0,manual_sims


---
## Cell 5: Export Reference Facets

The reference facets file contains unique facet geometry (centroids + areas)
from the first loaded case. This is used by the API for CSAR computation.

In [6]:
# Use the first loaded case as the reference geometry
ref_case = loaded_cases[0]

# Build reference facets DataFrame (unique facets, no insertion_depth column)
ref_facets_df = pd.DataFrame({
    'face_id':    np.arange(ref_case.n_facets, dtype=np.int32),
    'centroid_x': ref_case.centroids[:, 0],
    'centroid_y': ref_case.centroids[:, 1],
    'centroid_z': ref_case.centroids[:, 2],
    'facet_area': ref_case.areas,
})

# Save reference facets
ref_facets_df.to_csv(REFERENCE_FACETS_CSV, index=False)
print(f'✓ Reference facets saved: {REFERENCE_FACETS_CSV}')
print(f'  {len(ref_facets_df):,} facets')
print(f'  Z range: {ref_facets_df["centroid_z"].min():.1f} → '
      f'{ref_facets_df["centroid_z"].max():.1f} mm')
print(f'  Total surface area: {ref_facets_df["facet_area"].sum():.1f} mm²')
print()
print('This file will be used by the LibreChat surrogate tools:')
print('  compute_csar_vs_depth(), evaluate_contact_pressure()')
ref_facets_df.head()

✓ Reference facets saved: /workspace/data/surrogate/training/reference_facets.csv
  23,734 facets
  Z range: -0.3 → 402.4 mm
  Total surface area: 5752.7 mm²

This file will be used by the LibreChat surrogate tools:
  compute_csar_vs_depth(), evaluate_contact_pressure()


,face_id,centroid_x,centroid_y,centroid_z,facet_area
0,0,-0.365146,-2.227906,2.832700,0.187719
1,1,1.075712,-2.374332,2.554793,0.177791
2,2,-0.841743,0.670698,0.307375,0.318930
3,3,-1.161272,0.368586,11.753905,0.156565
4,4,-1.239121,-0.130011,11.437157,0.177645


---
## Cell 6: Configure Surrogate Model Training

In [7]:
import sys
import yaml
import copy

# Load base config from surrogate-lab
BASE_CONFIG_PATH = SURROGATE_LAB_DIR / 'configs' / 'config.yaml'
with open(BASE_CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# Override data source and MLflow URI
cfg['data']['source'] = str(TRAINING_DIR) + '/'
cfg['data']['file_pattern'] = 'combined.csv'
cfg['mlflow']['tracking_uri'] = MLFLOW_URI
cfg['mlflow']['experiment_name'] = MLFLOW_EXPERIMENT
cfg['training']['checkpoint']['dir'] = str(MODELS_DIR) + '/'

# ── Customise architecture and training (edit as needed) ─────────────────────
# Adjust based on dataset size: larger datasets can support deeper networks.
n_training_samples = len(combined_df)

if n_training_samples > 500_000:
    cfg['model']['layers'] = [256, 256, 128, 64]
    cfg['training']['epochs'] = 2
    cfg['training']['batch_size'] = 512
elif n_training_samples > 100_000:
    cfg['model']['layers'] = [128, 128, 64]
    cfg['training']['epochs'] = 200
    cfg['training']['batch_size'] = 256
else:
    cfg['model']['layers'] = [64, 64, 32]
    cfg['training']['epochs'] = 150
    cfg['training']['batch_size'] = 128

# Save the resolved config next to the training data
RESOLVED_CONFIG_PATH = TRAINING_DIR / 'training_config.yaml'
with open(RESOLVED_CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('Training configuration:')
print(f'  Dataset:      {COMBINED_CSV} ({n_training_samples:,} rows)')
print(f'  Architecture: MLP {cfg["model"]["layers"]}')
print(f'  Epochs:       {cfg["training"]["epochs"]}')
print(f'  Batch size:   {cfg["training"]["batch_size"]}')
print(f'  MLflow URI:   {cfg["mlflow"]["tracking_uri"]}')
print(f'  Checkpoint:   {cfg["training"]["checkpoint"]["dir"]}')
print(f'  Config saved: {RESOLVED_CONFIG_PATH}')

Training configuration:
  Dataset:      /workspace/data/surrogate/training/combined.csv (4,272,120 rows)
  Architecture: MLP [256, 256, 128, 64]
  Epochs:       2
  Batch size:   512
  MLflow URI:   http://mlflow:5000
  Checkpoint:   /workspace/data/surrogate/models/latest/
  Config saved: /workspace/data/surrogate/training/training_config.yaml


---
## Cell 7: Train the Surrogate Model

This cell runs the full training pipeline. Training time depends on dataset size and hardware:
- CPU (small dataset ~50k rows): ~2–5 minutes
- CPU (large dataset ~500k rows): ~15–30 minutes
- GPU: 5–10× faster

In [8]:
import mlflow
from src.data.loader import load_simulation_data
from src.data.schema import validate
from src.features.engineer import FeaturePipeline, build_xy
from src.features.splitter import split_data
from src.models.factory import build_model
from src.training.trainer import train
from src.evaluation.metrics import evaluate_model
from src.evaluation.visualization import (
    plot_predicted_vs_actual,
    plot_residuals,
    plot_training_curves,
)

print("Loading training data ...")
df_train = load_simulation_data(cfg, path=str(COMBINED_CSV))
validate(df_train, cfg)
print(f"  Dataset: {df_train.shape}")

print("Building feature matrices ...")
X, y = build_xy(df_train, cfg)
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y, cfg)
print(f"  Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")

print("Fitting feature pipeline ...")
pipeline = FeaturePipeline(cfg)
X_train_s, y_train_s = pipeline.fit_transform(X_train, y_train)
X_val_s,   y_val_s   = pipeline.transform(X_val,   y_val)
X_test_s,  y_test_s  = pipeline.transform(X_test,  y_test)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
pipeline.save(str(MODELS_DIR))
print(f"  Scalers saved to: {MODELS_DIR}")

print("Building model ...")
input_dim = X_train.shape[1]
model = build_model(input_dim=input_dim, cfg=cfg)
print(f"  Architecture: input_dim={input_dim} → {cfg['model']['layers']} → 1")

print()
print("Training ...")
print(f"  MLflow experiment: {MLFLOW_EXPERIMENT} @ {MLFLOW_URI}")
print()

model, active_mlflow_run_id = train(
    model, X_train_s, y_train_s,
    X_val_s, y_val_s,
    cfg,
    run_name=RUN_NAME,
)

print()
print("Training complete!")
print(f"  MLflow run ID: {active_mlflow_run_id}")


Loading training data ...
06:59:47 | INFO | src.data.loader | Loaded combined.csv                              rows=4272120


INFO:src.data.loader:Loaded combined.csv                              rows=4272120


06:59:47 | INFO | src.data.schema | Schema OK — shape=(4272120, 7)


INFO:src.data.schema:Schema OK — shape=(4272120, 7)


06:59:47 | INFO | src.data.loader | Dataset ready — 4272120 rows, 7 columns


...

INFO:src.training.trainer:MLflow run d81d47f3 started on cpu
/usr/local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


07:01:25 | INFO | src.training.trainer | Epoch    1/2  train=0.001213  val=0.000778  [95.8s]


INFO:src.training.trainer:Epoch    1/2  train=0.001213  val=0.000778  [95.8s]


🏃 View run smiling-asp-643 at: http://mlflow:5000/#/experiments/464447281740747989/runs/d81d47f31fb64a6fa3f7b01b8ffe0bc7
🧪 View experiment at: http://mlflow:5000/#/experiments/464447281740747989


PermissionError: [Errno 13] Permission denied: '/mlruns'

---
## Cell 8: Evaluate Model Performance

In [ ]:
import torch
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

print('Evaluating on test set ...')
metrics = evaluate_model(model, X_test_s, y_test_s, pipeline, device_str=device)
print()
print('Test Set Metrics (in original units [MPa]):')
print(f'  RMSE: {metrics["rmse"]:.6f} MPa')
print(f'  MAE:  {metrics["mae"]:.6f} MPa')
print(f'  R²:   {metrics["r2"]:.4f}')
print()

if metrics['r2'] >= 0.95:
    print('✓ Excellent fit (R² ≥ 0.95) — model ready for deployment')
elif metrics['r2'] >= 0.85:
    print('✓ Good fit (R² ≥ 0.85) — model usable but consider more data')
else:
    print('⚠ Moderate fit (R² < 0.85) — consider: more data, deeper network')

# Generate predictions for plots
model.eval()
with torch.no_grad():
    y_pred_s = model(torch.from_numpy(X_test_s).to(device)).cpu().numpy()
y_pred = pipeline.inverse_transform_y(y_pred_s)
y_true = pipeline.inverse_transform_y(y_test_s)

fig1 = plot_predicted_vs_actual(y_true, y_pred,
                                 title='Surrogate: Predicted vs Actual Contact Pressure',
                                 save_path=str(RESULTS_DIR / 'pred_vs_actual.png'))
plt.show()

fig2 = plot_residuals(y_true, y_pred,
                       save_path=str(RESULTS_DIR / 'residuals.png'))
plt.show()

print(f'Plots saved to: {RESULTS_DIR}')

In [ ]:
# ── Model Report: Presentation-Ready Plots & Metrics ─────────────────────────
# Generates a full diagnostic suite saved as individual PNGs in RESULTS_DIR.
# All figures use a clean, consistent style suitable for slides/papers.

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import mlflow
from mlflow.tracking import MlflowClient
from pathlib import Path

# ── Style ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.dpi": 150,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
})
PALETTE = {"primary": "#2563EB", "accent": "#DC2626", "neutral": "#6B7280", "ok": "#16A34A"}


# ── 1. Fetch training curves from MLflow ─────────────────────────────────────
mlflow.set_tracking_uri(MLFLOW_URI)
client = MlflowClient()

train_losses, val_losses = [], []
if active_mlflow_run_id:
    for metric_key, store in [("train_loss", train_losses), ("val_loss", val_losses)]:
        history = client.get_metric_history(active_mlflow_run_id, metric_key)
        store.extend(h.value for h in sorted(history, key=lambda h: h.step))
    print(f"Fetched {len(train_losses)} epochs of training history from MLflow")
else:
    print("No MLflow run ID — training curves unavailable")


# ── 2. Rebuild predictions if not already in scope ───────────────────────────
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval()
with torch.no_grad():
    y_pred_s = model(torch.from_numpy(X_test_s).to(device)).cpu().numpy()
y_pred = pipeline.inverse_transform_y(y_pred_s)
y_true = pipeline.inverse_transform_y(y_test_s)
residuals = y_pred.ravel() - y_true.ravel()

# Also grab insertion_depth from the unscaled test set (first feature = index 4 in cfg)
feat_names = cfg["features"]["inputs"]
depth_idx = feat_names.index("insertion_depth") if "insertion_depth" in feat_names else None


# ── Helper ────────────────────────────────────────────────────────────────────
def save(fig, name):
    p = RESULTS_DIR / name
    fig.savefig(p)
    print(f"  saved → {p.name}")
    return p


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 1: Learning Curves
# ══════════════════════════════════════════════════════════════════════════════
if train_losses:
    fig, ax = plt.subplots(figsize=(9, 4))
    epochs_range = range(1, len(train_losses) + 1)
    ax.plot(epochs_range, train_losses, color=PALETTE["primary"], lw=1.8, label="Train MSE")
    ax.plot(epochs_range, val_losses,   color=PALETTE["accent"],  lw=1.8, label="Val MSE", linestyle="--")
    best_epoch = int(np.argmin(val_losses)) + 1
    ax.axvline(best_epoch, color=PALETTE["neutral"], linestyle=":", lw=1.2,
               label=f"Best epoch ({best_epoch})")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss (scaled)")
    ax.set_title("Training & Validation Loss")
    ax.legend()
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    save(fig, "report_learning_curves.png")
    plt.show()
    print()


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 2: Predicted vs Actual  (publication scatter)
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(6, 6))
lo = min(y_true.min(), y_pred.min())
hi = max(y_true.max(), y_pred.max())
ax.scatter(y_true.ravel(), y_pred.ravel(),
           alpha=0.25, s=6, color=PALETTE["primary"], rasterized=True, label="Test samples")
ax.plot([lo, hi], [lo, hi], color=PALETTE["accent"], lw=1.5, linestyle="--", label="Ideal (y=x)")
ax.set_xlabel("Actual Contact Pressure [MPa]")
ax.set_ylabel("Predicted Contact Pressure [MPa]")
ax.set_title(f"Predicted vs Actual  (R² = {metrics['r2']:.4f})")
ax.legend()
ax.grid(True, alpha=0.2)
fig.tight_layout()
save(fig, "report_pred_vs_actual.png")
plt.show()
print()


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 3: Residual Diagnostics (2-panel)
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# 3a — Residuals vs Predicted
axes[0].scatter(y_pred.ravel(), residuals,
                alpha=0.2, s=5, color=PALETTE["primary"], rasterized=True)
axes[0].axhline(0, color=PALETTE["accent"], lw=1.5, linestyle="--")
p95 = np.percentile(np.abs(residuals), 95)
axes[0].axhline( p95, color=PALETTE["neutral"], lw=1, linestyle=":", label=f"±95th pct ({p95:.4f})")
axes[0].axhline(-p95, color=PALETTE["neutral"], lw=1, linestyle=":")
axes[0].set_xlabel("Predicted [MPa]")
axes[0].set_ylabel("Residual (pred − actual) [MPa]")
axes[0].set_title("Residuals vs Predicted")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.2)

# 3b — Residual histogram
axes[1].hist(residuals, bins=60, color=PALETTE["primary"], edgecolor="white", alpha=0.85)
axes[1].axvline(0, color=PALETTE["accent"], lw=1.5, linestyle="--", label="Zero")
axes[1].axvline(residuals.mean(), color=PALETTE["ok"], lw=1.5, linestyle="-",
                label=f"Mean = {residuals.mean():.5f}")
axes[1].set_xlabel("Residual [MPa]")
axes[1].set_ylabel("Count")
axes[1].set_title("Residual Distribution")
axes[1].legend()
axes[1].grid(True, alpha=0.2)

fig.suptitle("Residual Diagnostics", fontsize=14, y=1.01)
fig.tight_layout()
save(fig, "report_residuals.png")
plt.show()
print()


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 4: Error by Insertion Depth  (if depth feature available)
# ══════════════════════════════════════════════════════════════════════════════
if depth_idx is not None:
    # Inverse-transform the depth from scaled X_test
    X_test_orig = pipeline.x_scaler.inverse_transform(X_test_s)
    depths_test  = X_test_orig[:, depth_idx]

    # Bin into equal-width depth bands
    n_bins = 8
    depth_bins = np.linspace(depths_test.min(), depths_test.max(), n_bins + 1)
    bin_labels  = [f"{depth_bins[i]:.0f}–{depth_bins[i+1]:.0f}" for i in range(n_bins)]
    bin_indices = np.digitize(depths_test, depth_bins) - 1
    bin_indices = np.clip(bin_indices, 0, n_bins - 1)

    bin_rmse, bin_mae, bin_r2, bin_n = [], [], [], []
    for b in range(n_bins):
        mask = bin_indices == b
        if mask.sum() < 5:
            bin_rmse.append(np.nan); bin_mae.append(np.nan)
            bin_r2.append(np.nan);   bin_n.append(0)
        else:
            yt = y_true.ravel()[mask]; yp = y_pred.ravel()[mask]
            from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
            bin_rmse.append(np.sqrt(mean_squared_error(yt, yp)))
            bin_mae.append(mean_absolute_error(yt, yp))
            r2_val = r2_score(yt, yp)
            bin_r2.append(r2_val if np.isfinite(r2_val) else np.nan)
            bin_n.append(mask.sum())

    x_pos = np.arange(n_bins)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

    for ax, values, ylabel, color in [
        (axes[0], bin_rmse, "RMSE [MPa]",  PALETTE["primary"]),
        (axes[1], bin_mae,  "MAE [MPa]",   PALETTE["accent"]),
        (axes[2], bin_r2,   "R²",           PALETTE["ok"]),
    ]:
        bars = ax.bar(x_pos, values, color=color, alpha=0.8, width=0.65)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(bin_labels, rotation=35, ha="right", fontsize=8)
        ax.set_ylabel(ylabel)
        ax.set_xlabel("Insertion Depth Band [mm]")
        ax.grid(True, axis="y", alpha=0.25)

    axes[2].axhline(1.0, color=PALETTE["neutral"], lw=1, linestyle="--")
    fig.suptitle("Model Performance by Insertion Depth Band", fontsize=14)
    fig.tight_layout()
    save(fig, "report_error_by_depth.png")
    plt.show()
    print()
else:
    print("Depth feature not found in config — skipping depth-band plot")


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 5: Summary Metrics Card
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.metrics import mean_squared_error, mean_absolute_error
abs_err = np.abs(residuals)
pct_err = np.abs(residuals) / (np.abs(y_true.ravel()) + 1e-10) * 100.0

summary_rows = [
    ("RMSE",                  f"{metrics['rmse']:.6f}",     "MPa"),
    ("MAE",                   f"{metrics['mae']:.6f}",      "MPa"),
    ("R²",                    f"{metrics['r2']:.4f}",       "—"),
    ("Max absolute error",    f"{abs_err.max():.6f}",       "MPa"),
    ("95th pct |error|",      f"{np.percentile(abs_err,95):.6f}", "MPa"),
    ("Mean % error",          f"{pct_err.mean():.2f}",      "%"),
    ("95th pct % error",      f"{np.percentile(pct_err,95):.2f}", "%"),
    ("Test samples",          f"{len(y_true.ravel()):,}",   "—"),
]

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.axis("off")
col_labels = ["Metric", "Value", "Unit"]
table = ax.table(
    cellText=summary_rows,
    colLabels=col_labels,
    loc="center",
    cellLoc="left",
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.55)

# Header styling
for col in range(3):
    table[0, col].set_facecolor("#1E3A5F")
    table[0, col].set_text_props(color="white", fontweight="bold")

# Alternating row colours
for row in range(1, len(summary_rows) + 1):
    bg = "#F0F4FF" if row % 2 == 0 else "white"
    for col in range(3):
        table[row, col].set_facecolor(bg)

# Highlight R² row green/amber/red
r2_row = 3  # 1-based (row 0 = header)
r2_color = PALETTE["ok"] if metrics["r2"] >= 0.95 else ("#F59E0B" if metrics["r2"] >= 0.85 else PALETTE["accent"])
for col in range(3):
    table[r2_row, col].set_facecolor(r2_color)
    table[r2_row, col].set_text_props(color="white", fontweight="bold")

ax.set_title("Surrogate Model — Test Set Metrics", fontsize=13, pad=12, fontweight="bold")
fig.tight_layout()
save(fig, "report_metrics_card.png")
plt.show()
print()

print("=" * 60)
print("Report complete. All plots saved to:")
print(f"  {RESULTS_DIR}")
print()
for row in summary_rows:
    print(f"  {row[0]:30s}  {row[1]:>12s}  {row[2]}")

---
## Cell 9: Deploy Model Artifacts

Copies the trained model weights and config to the shared directory so the
LibreChat API can use them immediately (no restart required).

In [ ]:
import shutil
import torch
import mlflow
from mlflow.tracking import MlflowClient
import yaml, datetime

# ── 1. Save model weights ─────────────────────────────────────────────────────
model_weights_path = MODELS_DIR / "best_model.pt"
torch.save(model.state_dict(), model_weights_path)
print(f"✓ Model weights saved: {model_weights_path}")

for scaler_file in ["x_scaler.pkl", "y_scaler.pkl"]:
    if (MODELS_DIR / scaler_file).exists():
        print(f"✓ Scaler present:       {MODELS_DIR / scaler_file}")
    else:
        print(f"✗ Scaler MISSING:       {MODELS_DIR / scaler_file}")

config_dst = MODELS_DIR / "config.yaml"
shutil.copy(RESOLVED_CONFIG_PATH, config_dst)
print(f"✓ Config saved:         {config_dst}")

required = ["best_model.pt", "x_scaler.pkl", "y_scaler.pkl", "config.yaml"]
all_present = all((MODELS_DIR / f).exists() for f in required)
if not all_present:
    missing = [f for f in required if not (MODELS_DIR / f).exists()]
    raise RuntimeError(f"Missing model artifacts: {missing}")
print()
print(f"✓ All artifacts present at: {MODELS_DIR}")

# ── 2. Log extra metadata to the active MLflow run ───────────────────────────
mlflow.set_tracking_uri(MLFLOW_URI)
client = MlflowClient()

if active_mlflow_run_id:
    # Tag with data provenance
    sources_tag = " | ".join(str(s) for s in SIMULATION_SOURCES)
    client.set_tag(active_mlflow_run_id, "simulation_sources",  sources_tag)
    client.set_tag(active_mlflow_run_id, "n_cases_loaded",      str(len(loaded_cases)))
    client.set_tag(active_mlflow_run_id, "n_training_rows",     str(len(combined_df)))
    client.set_tag(active_mlflow_run_id, "artifact_dir",        str(MODELS_DIR))
    client.set_tag(active_mlflow_run_id, "deployed_at",
                   datetime.datetime.now().isoformat(timespec="seconds"))

    # Log model artifacts into the run
    with mlflow.start_run(run_id=active_mlflow_run_id):
        mlflow.log_artifacts(str(MODELS_DIR), artifact_path="model_artifacts")
        mlflow.log_artifact(str(COMBINED_CSV),          artifact_path="data")
        mlflow.log_artifact(str(REFERENCE_FACETS_CSV),  artifact_path="data")

    print(f"✓ Artifacts logged to MLflow run: {active_mlflow_run_id}")
else:
    print("⚠ No active MLflow run ID found — skipping registry step.")
    print("  (train() may not have returned a run_id; check src/training/trainer.py)")

print()
print("The surrogate model is now available in LibreChat:")
print("  • compute_csar_vs_depth()")
print("  • evaluate_contact_pressure()")
print("  • predict_vtp_contact_pressure()")


---
## Cell 10b: MLflow Model Registry — Version Management

Use this cell to:
* **Register** the just-trained model under a versioned name in the MLflow Model Registry.
* **Promote** a version to the `Production` alias so LibreChat picks it up automatically.
* **List** all available versions and their status.

Run this cell after Cell 9 (Deploy).


In [ ]:
import mlflow
from mlflow.tracking import MlflowClient
from mlflow.exceptions import MlflowException
import pandas as pd

mlflow.set_tracking_uri(MLFLOW_URI)
client = MlflowClient()

# ── Settings ──────────────────────────────────────────────────────────────────
MODEL_REGISTRY_NAME = "surrogate-contact-pressure"   # registry model name
# Alias that the LibreChat / API code reads to find the production model:
PRODUCTION_ALIAS    = "production"

# ── Register the model ────────────────────────────────────────────────────────
if active_mlflow_run_id:
    artifact_uri = f"runs:/{active_mlflow_run_id}/model_artifacts"
    try:
        mv = mlflow.register_model(artifact_uri, MODEL_REGISTRY_NAME)
        new_version = mv.version
        print(f"✓ Registered '{MODEL_REGISTRY_NAME}' version {new_version}")
        print(f"  Run:          {active_mlflow_run_id}")
        print(f"  Artifact URI: {artifact_uri}")
    except MlflowException as e:
        print(f"✗ Registration failed: {e}")
        new_version = None
else:
    print("No active_mlflow_run_id — skipping registration.")
    new_version = None

# ── Promote to production ─────────────────────────────────────────────────────
if new_version is not None:
    PROMOTE = True   # ← set to False to register without promoting
    if PROMOTE:
        client.set_registered_model_alias(
            MODEL_REGISTRY_NAME, PRODUCTION_ALIAS, new_version
        )
        print(f"✓ Version {new_version} aliased as '{PRODUCTION_ALIAS}'")
    else:
        print(f"  (not promoted — set PROMOTE = True to make this the production model)")

# ── Show all versions ─────────────────────────────────────────────────────────
print()
print(f"All versions of '{MODEL_REGISTRY_NAME}':")
print("-" * 72)

try:
    versions = client.search_model_versions(f"name='{MODEL_REGISTRY_NAME}'")
    rows = []
    for v in sorted(versions, key=lambda x: int(x.version)):
        # Fetch aliases for this version
        aliases = [a for a, ver in
                   client.get_registered_model(MODEL_REGISTRY_NAME).aliases.items()
                   if ver == v.version]
        run_tags = {}
        try:
            run = client.get_run(v.run_id)
            run_tags = run.data.tags
        except Exception:
            pass
        rows.append({
            "Version": v.version,
            "Status":  v.status,
            "Aliases": ", ".join(aliases) if aliases else "—",
            "Run ID":  v.run_id[:8] + "...",
            "n_rows":  run_tags.get("n_training_rows", "?"),
            "n_cases": run_tags.get("n_cases_loaded",  "?"),
            "deployed": run_tags.get("deployed_at",     "?"),
        })
    if rows:
        df_versions = pd.DataFrame(rows).set_index("Version")
        print(df_versions.to_string())
    else:
        print("  (no versions registered yet)")
except MlflowException as e:
    print(f"  Could not list versions: {e}")

print()
print(f"To load the production model in code:")
print(f'  model_uri = f"models:/{MODEL_REGISTRY_NAME}@{PRODUCTION_ALIAS}"')
print(f'  artifacts = mlflow.artifacts.download_artifacts(model_uri)')


---
## Cell 10: Quick Surrogate Validation

Run a quick CSAR computation to verify the deployed model works.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys

# Add the simulation-api-tool package to path
api_tool_dir = REPO_ROOT
if str(api_tool_dir) not in sys.path:
    sys.path.insert(0, str(api_tool_dir))

try:
    from digital_twin_ui.surrogate.predictor import SurrogatePredictor, is_model_available
    from digital_twin_ui.surrogate.csar_engine import CSAREngine, build_insertion_depths

    print('Testing surrogate predictor ...')

    if not is_model_available(MODELS_DIR):
        print('Model artifacts not found. Ensure Cell 9 completed successfully.')
    else:
        predictor = SurrogatePredictor(MODELS_DIR)
        engine = CSAREngine(predictor)

        # Load reference facets
        ref_df = pd.read_csv(REFERENCE_FACETS_CSV)

        # Z range for the reference geometry
        z_min = ref_df['centroid_z'].min()
        z_max = ref_df['centroid_z'].max()
        z_mid = (z_min + z_max) / 2

        z_bands = [
            {'zmin': z_min, 'zmax': z_mid, 'label': 'distal_half'},
            {'zmin': z_mid, 'zmax': z_max, 'label': 'proximal_half'},
        ]

        depths = build_insertion_depths(max_depth_mm=200, step_mm=10)
        csar_df = engine.compute_csar_vs_depth(ref_df, depths, z_bands)

        # Plot
        fig, ax = plt.subplots(figsize=(10, 5))
        for label, group in csar_df.groupby('band_label'):
            g = group.sort_values('insertion_depth_mm')
            ax.plot(g['insertion_depth_mm'], g['csar'], label=label, lw=2)

        ax.set_xlabel('Insertion Depth [mm]', fontsize=12)
        ax.set_ylabel('CSAR', fontsize=12)
        ax.set_title('Surrogate Model — CSAR vs Insertion Depth', fontsize=13)
        ax.legend()
        ax.set_ylim(0, 1.05)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()

        plot_path = RESULTS_DIR / 'csar_validation.png'
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        plt.show()

        print(f'\n✓ Surrogate validation plot saved: {plot_path}')
        print('\nSample CSAR values (distal_half band):')
        sample = csar_df[csar_df['band_label']=='distal_half'].sort_values('insertion_depth_mm')
        print(sample[['insertion_depth_mm','csar','n_contact_facets','max_cp_MPa']].head(10).to_string(index=False))

except ImportError as e:
    print(f'Import error: {e}')
    print('The digital_twin_ui package may not be on PYTHONPATH when running locally.')
    print('This is OK — the model will still work via the Docker API.')

---
## Cell 11: Export VTP Files (optional)

Export VTP files from the loaded simulation cases for ParaView visualization
and for testing the `predict_vtp_contact_pressure` API endpoint.

In [ ]:
# Export VTP files to data/surrogate/results/ for ParaView and API testing
VTP_OUTPUT_DIR = RESULTS_DIR

print('Exporting VTP files ...')
exported_vtps = []

for sc in loaded_cases[:2]:  # Export first 2 cases to avoid large disk usage
    case_vtp_dir = VTP_OUTPUT_DIR / f'{sc._label}_vtp'
    try:
        pvd_path = sc.export_vtp(case_vtp_dir)
        vtp_files = list(case_vtp_dir.glob('*.vtp'))
        exported_vtps.extend(vtp_files)
        print(f'  ✓ {sc._label}: {len(vtp_files)} VTP files → {case_vtp_dir}')
        print(f'    PVD collection: {pvd_path}')
    except Exception as e:
        print(f'  ✗ {sc._label}: {e}')

print()
if exported_vtps:
    print(f'VTP files are accessible at: {VTP_OUTPUT_DIR}')
    print('In LibreChat, use container path:')
    print(f'  /app/surrogate_data/results/<case_name>_vtp/<case_name>_t0000.vtp')
    print()
    print('Example LibreChat command:')
    first_vtp = exported_vtps[0]
    container_path = f'/app/surrogate_data/results/{first_vtp.parent.name}/{first_vtp.name}'
    print(f'  predict_vtp_contact_pressure({container_path!r}, insertion_depth_mm=100.0)')

---
## Summary

If all cells ran successfully, the following artifacts have been created:

| File | Location | Purpose |
|------|----------|---------|
| `combined.csv` | `data/surrogate/training/` | Training dataset |
| `reference_facets.csv` | `data/surrogate/training/` | Facet geometry for CSAR |
| `best_model.pt` | `data/surrogate/models/latest/` | Trained model weights |
| `x_scaler.pkl` | `data/surrogate/models/latest/` | Feature scaler |
| `y_scaler.pkl` | `data/surrogate/models/latest/` | Target scaler |
| `config.yaml` | `data/surrogate/models/latest/` | Model architecture config |

The model is now available in **LibreChat** via the surrogate tools:
- `list_surrogate_models()` — verify `latest_available=true`
- `compute_csar_vs_depth(z_bands=[...])` — CSAR analysis
- `evaluate_contact_pressure(insertion_depths_mm=[...])` — pressure profile
- `predict_vtp_contact_pressure(vtp_path, depth)` — annotate VTP files

**JupyterLab** is accessible at `http://localhost:8888?token=dtui-jupyter`